# EEG-Based MDD Classification: Spectral Power + Alpha Asymmetry → SVM / LR / NB
## 33 MDD · 30 HC · 19 Channels · Eyes-Closed & Eyes-Open · Monte Carlo 10-Fold CV × 100

**Methodology pipeline:**
```
Single EEG folder  (MDD__.edf / H__.edf)
         ↓
Preprocessing
  → Load 19 channels (10-20 system, exact -LE suffix names)
  → Bandpass [0.5 – 45 Hz]  +  Notch [50 Hz]
  → Select 120 s of clean signal per subject
  → Eyes-Closed (EC) and Eyes-Open (EO) handled as separate condition tags
         ↓
Feature Extraction  (per subject, per condition)
  ├─ Spectral Power (Welch PSD)
  │    delta (0.5–4 Hz), theta (4–8 Hz), alpha (8–13 Hz), beta (13–30 Hz)
  │    19 channels × 4 bands = 76 features
  └─ Alpha Interhemispheric Asymmetry
       (Right – Left) / (Right + Left)  per hemisphere pair
       5 regions × 1 pair each = 5 features
         ↓
Concatenate → Z-score normalization
         ↓
Feature Selection  (AUC-based ranking)
  Rank all features by individual AUC vs MDD label
  Evaluate top-5 / top-10 / top-15 / top-19 subsets
         ↓
Three Classifiers compared
  ├─ Logistic Regression (LR)
  ├─ SVM  linear kernel
  └─ Naïve Bayes (GaussianNB)
         ↓
Monte Carlo 10-Fold CV × 100 repeats
  Normalization + AUC ranking fitted on TRAIN fold only
         ↓
Accuracy · Sensitivity · Specificity · F1 · AUC-ROC
```


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 1 — Install dependencies
# ═══════════════════════════════════════════════════════════════════════════
!pip install mne numpy scipy scikit-learn matplotlib seaborn pandas tqdm


In [3]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 2 — Imports
# ═══════════════════════════════════════════════════════════════════════════
import os, warnings, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from itertools import product

import mne
from scipy.signal import welch

from sklearn.svm              import SVC
from sklearn.linear_model     import LogisticRegression
from sklearn.naive_bayes      import GaussianNB
from sklearn.preprocessing    import StandardScaler
from sklearn.model_selection  import StratifiedKFold
from sklearn.metrics          import (accuracy_score, f1_score,
                                       roc_auc_score, confusion_matrix,
                                       ConfusionMatrixDisplay, roc_curve)
from sklearn.base             import clone

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
SEED = 42
np.random.seed(SEED)

print(f"MNE     : {mne.__version__}")
print(f"sklearn : {__import__('sklearn').__version__}")
print("All imports OK.")


MNE     : 1.11.0
sklearn : 1.7.2
All imports OK.


In [5]:
!dir

 Volume in drive G is New Volume
 Volume Serial Number is 34DF-EBC7

 Directory of g:\For Job\EEG project

2026-03-15  12:29 AM    <DIR>          .
2026-03-07  03:00 PM    <DIR>          ..
2026-03-08  10:05 AM                45 .gitignore
2026-03-14  06:44 PM    <DIR>          .ipynb_checkpoints
2026-03-08  03:12 PM    <DIR>          .venv
2026-03-14  09:19 PM            73,927 ECG_MDD_ThreeBranch_CNN (1).ipynb
2026-03-10  08:57 AM    <DIR>          EEG data
2026-03-14  07:28 PM            65,042 EEG_MDD_AsymmetryDFA_SVM (1).ipynb
2026-03-14  08:36 PM            73,742 EEG_MDD_CKSVM_Pipeline (2).ipynb
2026-03-13  10:33 PM            69,761 EEG_MDD_MultiBranch_Pipeline.ipynb
2026-03-15  12:32 AM            62,215 EEG_MDD_SpectralAsymmetry_SVM.ipynb
2026-03-14  07:23 PM           150,326 EEG_v3_corrected_flow.ipynb
2026-03-07  03:00 PM    <DIR>          main
2026-03-10  08:43 AM                 0 main.py
2026-03-11  10:40 AM    <DIR>          paper
2026-03-14  07:23 PM             1,189

In [6]:

import os
EEG_DIR = 'EEG data'

if not os.path.isdir(EEG_DIR):
    raise FileNotFoundError(f'Folder not found: {EEG_DIR}')

all_edf = sorted([f for f in os.listdir(EEG_DIR) if f.endswith('.edf')])
mdd_edf = [f for f in all_edf if os.path.splitext(f)[0].startswith('MDD')]
hc_edf  = [f for f in all_edf if os.path.splitext(f)[0].startswith('H')]
other   = [f for f in all_edf if f not in mdd_edf and f not in hc_edf]

print(f'EEG folder  : {EEG_DIR}')
print(f'Total .edf  : {len(all_edf)}')
print(f'MDD files   : {len(mdd_edf)}   (need = 33)')
print(f'HC  files   : {len(hc_edf)}   (need = 30)')
if other:
    print(f'UNMATCHED   : {other}  ← check prefix')
print(f'Sample names: {all_edf[:4]}')


EEG folder  : EEG data
Total .edf  : 181
MDD files   : 95   (need = 33)
HC  files   : 84   (need = 30)
UNMATCHED   : ['6921143_H S15 EO.edf', '6921959_H S15 EO.edf']  ← check prefix
Sample names: ['6921143_H S15 EO.edf', '6921959_H S15 EO.edf', 'H S1 EC.edf', 'H S1 EO.edf']


In [7]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 4 — Global Configuration
# ═══════════════════════════════════════════════════════════════════════════
CFG = {
    # ── Paths ──────────────────────────────────────────────────────────
    'eeg_dir'      : 'EEG data',
    'proc_dir'     : 'processed',
    'results_dir'  : 'results',
    'mdd_prefix'   : 'MDD',
    'hc_prefix'    : 'H',

    # ── 19 channels — exact names from your EDF files ──────────────────
    'channels_19' : [
        'EEG Fp1-LE', 'EEG F3-LE',  'EEG C3-LE',  'EEG P3-LE',
        'EEG O1-LE',  'EEG F7-LE',  'EEG T3-LE',  'EEG T5-LE',
        'EEG Fz-LE',  'EEG Fp2-LE', 'EEG F4-LE',  'EEG C4-LE',
        'EEG P4-LE',  'EEG O2-LE',  'EEG F8-LE',  'EEG T4-LE',
        'EEG T6-LE',  'EEG Cz-LE',  'EEG Pz-LE'
    ],

    # ── Alpha asymmetry pairs (left_ch, right_ch) per brain region ─────
    # Based on standard EEG interhemispheric asymmetry literature
    'asym_pairs' : {
        'frontal'    : ('EEG F3-LE',  'EEG F4-LE'),
        'temporal'   : ('EEG T3-LE',  'EEG T4-LE'),
        'parietal'   : ('EEG P3-LE',  'EEG P4-LE'),
        'occipital'  : ('EEG O1-LE',  'EEG O2-LE'),
        'central'    : ('EEG C3-LE',  'EEG C4-LE'),
    },

    # ── Preprocessing ──────────────────────────────────────────────────
    'sfreq_target' : 256,
    'l_freq'       : 0.5,
    'h_freq'       : 45.0,
    'notch_freq'   : 50.0,
    'clean_sec'    : 120.0,     # 120 clean seconds per subject

    # ── Eyes-Closed / Eyes-Open condition ─────────────────────────────
    # If your EDF contains BOTH conditions in one file, set to 'both'.
    # The pipeline will split the recording in halves (EC first, EO second).
    # If each condition is a separate file, set to 'auto' — condition
    # is inferred from filename suffix (_EC / _EO) or treated as one condition.
    'condition_mode' : 'auto',   # 'auto' | 'split_halves' | 'ec_only' | 'eo_only'

    # ── Frequency bands ────────────────────────────────────────────────
    'bands' : {
        'delta' : (0.5,  4.0),
        'theta' : (4.0,  8.0),
        'alpha' : (8.0,  13.0),
        'beta'  : (13.0, 30.0),
    },

    # ── Feature selection ──────────────────────────────────────────────
    'feature_subsets' : [5, 10, 15, 19],   # top-N subsets to evaluate

    # ── Monte Carlo CV ─────────────────────────────────────────────────
    'n_repeats'    : 100,
    'k_folds'      : 10,
}

os.makedirs(CFG['proc_dir'],    exist_ok=True)
os.makedirs(CFG['results_dir'], exist_ok=True)

n_ch    = len(CFG['channels_19'])
n_band  = len(CFG['bands'])
n_asym  = len(CFG['asym_pairs'])
n_spec  = n_ch * n_band
n_total = n_spec + n_asym

print('Configuration loaded.')
print(f'  Channels        : {n_ch}')
print(f'  Clean duration  : {CFG["clean_sec"]} s per subject')
print(f'  Condition mode  : {CFG["condition_mode"]}')
print(f'\nExpected feature count:')
print(f'  Spectral power  : {n_ch} ch × {n_band} bands = {n_spec}')
print(f'  Alpha asymmetry : {n_asym} region pairs = {n_asym}')
print(f'  TOTAL           : {n_total}')
print(f'\nMonte Carlo CV  : {CFG["n_repeats"]} repeats × {CFG["k_folds"]}-fold')


Configuration loaded.
  Channels        : 19
  Clean duration  : 120.0 s per subject
  Condition mode  : auto

Expected feature count:
  Spectral power  : 19 ch × 4 bands = 76
  Alpha asymmetry : 5 region pairs = 5
  TOTAL           : 81

Monte Carlo CV  : 100 repeats × 10-fold


In [8]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 5 — Dataset Audit
# ═══════════════════════════════════════════════════════════════════════════
eeg_dir = CFG['eeg_dir']
all_edf = sorted([f for f in os.listdir(eeg_dir) if f.endswith('.edf')])
mdd_edf = [f for f in all_edf if os.path.splitext(f)[0].startswith(CFG['mdd_prefix'])]
hc_edf  = [f for f in all_edf if os.path.splitext(f)[0].startswith(CFG['hc_prefix'])]

print('═' * 62)
print('  DATASET AUDIT')
print('═' * 62)
print(f'  MDD files : {len(mdd_edf)}   (paper = 33)')
print(f'  HC  files : {len(hc_edf)}   (paper = 30)')
print(f'  Total     : {len(all_edf)}')

for sample_f, group in [(mdd_edf[0] if mdd_edf else None, 'MDD'),
                         (hc_edf[0]  if hc_edf  else None, 'HC')]:
    if not sample_f: continue
    raw = mne.io.read_raw_edf(os.path.join(eeg_dir, sample_f),
                               preload=False, verbose=False)
    print(f'\n  [{group}] {sample_f}')
    print(f'    sfreq      : {raw.info["sfreq"]} Hz')
    print(f'    n_channels : {len(raw.ch_names)}')
    print(f'    duration   : {raw.times[-1]:.1f} s  '
          f'(need ≥ {CFG["clean_sec"]:.0f} s)')
    print(f'    ch_names   : {raw.ch_names[:6]} ...')

    found   = [ch for ch in CFG['channels_19'] if ch in raw.ch_names]
    missing = [ch for ch in CFG['channels_19'] if ch not in raw.ch_names]
    print(f'    channels_19 found : {len(found)}/19')
    if missing:
        print(f'    MISSING  : {missing}')
        print(f'    ► Update CFG["channels_19"] to match actual names above')
    else:
        print(f'    ✓ All 19 channels confirmed')

    # Check asymmetry pair channels
    asym_missing = []
    for region, (l_ch, r_ch) in CFG['asym_pairs'].items():
        for ch in [l_ch, r_ch]:
            if ch not in raw.ch_names:
                asym_missing.append(ch)
    if asym_missing:
        print(f'    ASYM PAIRS MISSING: {asym_missing}')
    else:
        print(f'    ✓ All asymmetry pair channels confirmed')

print('═' * 62)


══════════════════════════════════════════════════════════════
  DATASET AUDIT
══════════════════════════════════════════════════════════════
  MDD files : 95   (paper = 33)
  HC  files : 84   (paper = 30)
  Total     : 181

  [MDD] MDD S1  EO.edf
    sfreq      : 256.0 Hz
    n_channels : 20
    duration   : 301.0 s  (need ≥ 120 s)
    ch_names   : ['EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE', 'EEG F7-LE'] ...
    channels_19 found : 19/19
    ✓ All 19 channels confirmed
    ✓ All asymmetry pair channels confirmed

  [HC] H S1 EC.edf
    sfreq      : 256.0 Hz
    n_channels : 22
    duration   : 300.0 s  (need ≥ 120 s)
    ch_names   : ['EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE', 'EEG F7-LE'] ...
    channels_19 found : 19/19
    ✓ All 19 channels confirmed
    ✓ All asymmetry pair channels confirmed
══════════════════════════════════════════════════════════════


In [9]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 6 — Preprocessing Function
#  Pipeline per subject:
#   1. Load EDF → pick 19 channels
#   2. Resample to 256 Hz if needed
#   3. Bandpass [0.5 – 45 Hz]  (zero-phase FIR, Hamming)
#   4. Notch filter [50 Hz]
#   5. Select 120 clean seconds  (first 120 s after filtering)
#   6. Condition split if needed (EC / EO)
# ═══════════════════════════════════════════════════════════════════════════

def preprocess_subject(filepath, cfg):
    """
    Preprocess one EDF file.

    Returns
    -------
    data     : dict  { 'ec': np.ndarray (n_ch, T),
                       'eo': np.ndarray (n_ch, T),
                       'combined': np.ndarray (n_ch, T) }
               T = clean_sec × sfreq_target
    ch_names : list of active channel names
    sfreq    : float
    """
    raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)

    # ── Pick 19 channels ──────────────────────────────────────────────
    available = [ch for ch in cfg['channels_19'] if ch in raw.ch_names]
    if len(available) < 10:
        print(f'  WARN {os.path.basename(filepath)}: '
              f'only {len(available)} channels found')
        return None, None, None
    raw.pick_channels(available)

    # ── Resample ──────────────────────────────────────────────────────
    if abs(raw.info['sfreq'] - cfg['sfreq_target']) > 1:
        raw.resample(cfg['sfreq_target'], npad='auto', verbose=False)
    sfreq = float(raw.info['sfreq'])

    # ── Bandpass [0.5–45 Hz] ──────────────────────────────────────────
    raw.filter(l_freq=cfg['l_freq'], h_freq=cfg['h_freq'],
               method='fir', fir_window='hamming', verbose=False)

    # ── Notch [50 Hz] ─────────────────────────────────────────────────
    raw.notch_filter(freqs=[cfg['notch_freq']], verbose=False)

    total_samples = raw.get_data().shape[1]
    clean_samples = int(cfg['clean_sec'] * sfreq)   # 120 × 256 = 30720

    if total_samples < clean_samples:
        print(f'  WARN {os.path.basename(filepath)}: '
              f'only {total_samples/sfreq:.1f}s available '
              f'(need {cfg["clean_sec"]}s)')
        clean_samples = total_samples

    data_all = raw.get_data()[:, :clean_samples].astype(np.float32)

    # ── Condition handling ────────────────────────────────────────────
    fname  = os.path.basename(filepath).lower()
    result = {}

    if cfg['condition_mode'] == 'split_halves':
        # Recording contains EC first, EO second (each 5 min)
        half = clean_samples // 2
        result['ec']       = data_all[:, :half]
        result['eo']       = data_all[:, half:half*2]
        result['combined'] = data_all

    elif cfg['condition_mode'] == 'ec_only':
        result['ec']       = data_all
        result['combined'] = data_all

    elif cfg['condition_mode'] == 'eo_only':
        result['eo']       = data_all
        result['combined'] = data_all

    else:   # 'auto' — infer from filename suffix
        if '_ec' in fname or 'eyesclosed' in fname:
            result['ec']       = data_all
            result['combined'] = data_all
        elif '_eo' in fname or 'eyesopen' in fname:
            result['eo']       = data_all
            result['combined'] = data_all
        else:
            # No condition tag found → treat as single condition
            result['combined'] = data_all

    return result, list(raw.ch_names), sfreq


# ── Smoke-test ─────────────────────────────────────────────────────────────
_files = sorted([f for f in os.listdir(CFG['eeg_dir']) if f.endswith('.edf')])
if _files:
    print(f'Smoke test: {_files[0]}')
    _d, _ch, _sf = preprocess_subject(
        os.path.join(CFG['eeg_dir'], _files[0]), CFG
    )
    if _d is not None:
        for cond, arr in _d.items():
            print(f'  [{cond}] shape={arr.shape}  '
                  f'({arr.shape[1]/int(_sf):.0f}s @ {int(_sf)}Hz)')
        print(f'  Channels ({len(_ch)}): {_ch[:4]} ...')


Smoke test: 6921143_H S15 EO.edf
  [combined] shape=(19, 30720)  (120s @ 256Hz)
  Channels (19): ['EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE'] ...


In [10]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 7 — Feature Extraction Functions
#  Two families:
#   1. Spectral band power  (Welch PSD, log-transformed)
#   2. Alpha interhemispheric asymmetry index
# ═══════════════════════════════════════════════════════════════════════════

# ─── 1. Spectral Band Power ───────────────────────────────────────────────

def compute_spectral_power(data, sfreq, bands, nperseg=None):
    """
    Log-transformed band power per channel × frequency band.
    Uses Welch's periodogram with Hann window.

    Parameters
    ----------
    data    : (n_ch, T)
    sfreq   : sampling frequency
    bands   : dict {name: (fmin, fmax)}

    Returns
    -------
    features   : 1-D array  (n_ch × n_bands,)
    feat_names : list of strings
    """
    nperseg = nperseg or min(data.shape[1], int(sfreq * 2))
    n_ch    = data.shape[0]
    features, feat_names = [], []

    for ch_i in range(n_ch):
        freqs, psd = welch(data[ch_i], fs=sfreq,
                            nperseg=nperseg, window='hann')
        for bname, (fmin, fmax) in bands.items():
            mask  = (freqs >= fmin) & (freqs < fmax)
            power = np.trapz(psd[mask], freqs[mask]) if mask.any() else 1e-12
            features.append(np.log1p(max(power, 1e-12)))   # log(1+power)
            feat_names.append(f'BP_{bname}_ch{ch_i:02d}')

    return np.array(features, dtype=np.float32), feat_names


# ─── 2. Alpha Interhemispheric Asymmetry ──────────────────────────────────

def compute_alpha_asymmetry(data, sfreq, asym_pairs, ch_names,
                             alpha_band=(8.0, 13.0), nperseg=None):
    """
    Interhemispheric asymmetry index for alpha band:
        AI = (Power_Right - Power_Left) / (Power_Right + Power_Left + ε)

    Positive AI → right-hemisphere dominance
    Negative AI → left-hemisphere dominance (common in MDD frontal alpha)

    Parameters
    ----------
    asym_pairs : dict { region_name: (left_ch_name, right_ch_name) }
    ch_names   : list of channel names matching data rows

    Returns
    -------
    features   : 1-D array  (n_regions,)
    feat_names : list of strings
    """
    nperseg   = nperseg or min(data.shape[1], int(sfreq * 2))
    ch_idx    = {ch: i for i, ch in enumerate(ch_names)}
    features, feat_names = [], []
    fmin, fmax = alpha_band

    for region, (l_ch, r_ch) in asym_pairs.items():
        if l_ch not in ch_idx or r_ch not in ch_idx:
            features.append(0.0)
            feat_names.append(f'Asym_alpha_{region}')
            continue

        li  = ch_idx[l_ch]
        ri  = ch_idx[r_ch]

        _, psd_l = welch(data[li], fs=sfreq, nperseg=nperseg, window='hann')
        _, psd_r = welch(data[ri], fs=sfreq, nperseg=nperseg, window='hann')
        freqs, _ = welch(data[li], fs=sfreq, nperseg=nperseg, window='hann')

        mask   = (freqs >= fmin) & (freqs < fmax)
        pow_l  = np.trapz(psd_l[mask], freqs[mask]) if mask.any() else 1e-12
        pow_r  = np.trapz(psd_r[mask], freqs[mask]) if mask.any() else 1e-12

        ai = (pow_r - pow_l) / (pow_r + pow_l + 1e-12)
        features.append(float(ai))
        feat_names.append(f'Asym_alpha_{region}')

    return np.array(features, dtype=np.float32), feat_names


# ─── Combined extractor ───────────────────────────────────────────────────

def extract_features(data, sfreq, ch_names, cfg):
    """
    Extract all features from a single (n_ch, T) data array.
    Returns (feature_vector_1D, feature_names_list).
    """
    f_spec, n_spec = compute_spectral_power(data, sfreq, cfg['bands'])
    f_asym, n_asym = compute_alpha_asymmetry(
        data, sfreq, cfg['asym_pairs'], ch_names,
        alpha_band=cfg['bands']['alpha']
    )
    return np.concatenate([f_spec, f_asym]), n_spec + n_asym


print('Feature extraction functions defined.')
n_ch   = len(CFG['channels_19'])
n_spec = n_ch * len(CFG['bands'])
n_asym = len(CFG['asym_pairs'])
print(f'  Spectral power  : {n_spec}  ({n_ch} ch × {len(CFG["bands"])} bands)')
print(f'  Alpha asymmetry : {n_asym}  ({n_asym} hemisphere pairs)')
print(f'  TOTAL           : {n_spec + n_asym}')


Feature extraction functions defined.
  Spectral power  : 76  (19 ch × 4 bands)
  Alpha asymmetry : 5  (5 hemisphere pairs)
  TOTAL           : 81


In [ ]:
# ── Cache ──────────────────────────────────────────────────────────────────
cache_F  = os.path.join(CFG['proc_dir'], 'F_all.npy')
cache_y  = os.path.join(CFG['proc_dir'], 'y_all.npy')
cache_fn = os.path.join(CFG['proc_dir'], 'feat_names.json')

if all(os.path.exists(p) for p in [cache_F, cache_y, cache_fn]):
    print('Loading cached feature matrix...')
    F_all      = np.load(cache_F)
    y_all      = np.load(cache_y)
    with open(cache_fn) as f:
        feat_names = json.load(f)
    print(f'  F_all={F_all.shape}, MDD={np.sum(y_all==1)}, HC={np.sum(y_all==0)}')
    
    # Validate cache — if empty, rebuild
    if F_all.size == 0 or feat_names is None:
        print('  ⚠ Cache corrupted or empty — rebuilding...')
        F_all, y_all, feat_names, ch_names = build_feature_matrix(
            CFG['eeg_dir'], CFG, condition='combined'
        )
        np.save(cache_F, F_all)
        np.save(cache_y, y_all)
        with open(cache_fn, 'w') as fh:
            json.dump(feat_names, fh)
        print('  Rebuilt and saved to disk.')
else:
    F_all, y_all, feat_names, ch_names = build_feature_matrix(
        CFG['eeg_dir'], CFG, condition='combined'
    )
    np.save(cache_F, F_all)
    np.save(cache_y, y_all)
    with open(cache_fn, 'w') as fh:
        json.dump(feat_names, fh)
    print('Saved to disk.')

if feat_names is None:
    raise ValueError('feat_names is None — check preprocessing output above')

n_spec_f = sum(1 for n in feat_names if n.startswith('BP_'))
n_asym_f = sum(1 for n in feat_names if n.startswith('Asym_'))
print(f'\nFeature breakdown: Spectral={n_spec_f} | Asymmetry={n_asym_f} | Total={len(feat_names)}')


Loading cached feature matrix...
  F_all=(0,), MDD=0, HC=0
  ⚠ Cache corrupted or empty — rebuilding...
  SKIP (unknown prefix): 6921143_H S15 EO.edf
  SKIP (unknown prefix): 6921959_H S15 EO.edf
Processing 179 subjects  (MDD=95, HC=84)


  Preprocess+Features:   1%|          | 1/179 [00:00<02:40,  1.11it/s]

  ERROR H S1 EC.edf: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


  Preprocess+Features:   1%|          | 2/179 [00:01<01:41,  1.75it/s]

  ERROR H S1 EO.edf: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


...

  ERROR MDD S19 EC.edf: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


  Preprocess+Features:  64%|██████▎   | 114/179 [00:47<00:40,  1.62it/s]

  ERROR MDD S19 EO.edf: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


  Preprocess+Features:  64%|██████▍   | 115/179 [00:48<00:47,  1.34it/s]

  ERROR MDD S19 TASK.edf: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 9 — AUC-Based Feature Ranking (global, for insight only)
#  NOTE: In the Monte Carlo CV (Cell 10), AUC ranking is re-fitted
#        on EACH TRAINING FOLD to prevent leakage.
#        This cell provides a global view of feature discriminability.
# ═══════════════════════════════════════════════════════════════════════════

def auc_feature_ranking(F, y):
    """
    Rank features by individual AUC (area under ROC curve).
    AUC > 0.5 means the feature is positively associated with MDD.
    AUC < 0.5 is flipped to 1-AUC (directionality does not matter for ranking).

    Returns
    -------
    ranked_idx  : indices sorted by AUC (highest first)
    auc_scores  : AUC score per feature
    """
    auc_scores = np.zeros(F.shape[1])
    for j in range(F.shape[1]):
        try:
            a = roc_auc_score(y, F[:, j])
            auc_scores[j] = max(a, 1.0 - a)   # direction-agnostic
        except Exception:
            auc_scores[j] = 0.5

    ranked_idx = np.argsort(auc_scores)[::-1]
    return ranked_idx, auc_scores


global_ranked_idx, global_auc_scores = auc_feature_ranking(F_all, y_all)

# ── Print top-20 features ──────────────────────────────────────────────────
print('Top 20 most discriminative features (global AUC ranking):')
print(f'  {"Rank":<5} {"Feature":<35} {"AUC":>6}')
print('  ' + '─'*48)
for rank, idx in enumerate(global_ranked_idx[:20]):
    print(f'  {rank+1:<5} {feat_names[idx]:<35} {global_auc_scores[idx]:.4f}')

# ── Feature family dominance ──────────────────────────────────────────────
top_n = 19
top_names = [feat_names[i] for i in global_ranked_idx[:top_n]]
n_bp_top  = sum(1 for n in top_names if n.startswith('BP_'))
n_as_top  = sum(1 for n in top_names if n.startswith('Asym_'))
print(f'\nTop-{top_n} composition: Spectral={n_bp_top}, Asymmetry={n_as_top}')

# ── AUC distribution plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(len(global_auc_scores)),
             np.sort(global_auc_scores)[::-1],
             color=['#FF9800' if s > 0.75 else '#90CAF9'
                    for s in np.sort(global_auc_scores)[::-1]],
             width=1.0, edgecolor='none')
axes[0].axhline(0.5, color='red', lw=1.0, linestyle='--', label='Chance')
axes[0].set_xlabel('Feature rank'); axes[0].set_ylabel('AUC')
axes[0].set_title('Feature AUC Distribution (all features)', fontweight='bold')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Top-10 named
top10_idx   = global_ranked_idx[:10]
top10_names = [feat_names[i].replace('EEG_','').replace('-LE','')[:22]
               for i in top10_idx]
top10_aucs  = global_auc_scores[top10_idx]
axes[1].barh(range(10)[::-1], top10_aucs,
              color=['#4CAF50' if 'Asym' in feat_names[i] else '#2196F3'
                     for i in top10_idx],
              edgecolor='black', alpha=0.85)
axes[1].set_yticks(range(10)[::-1])
axes[1].set_yticklabels(top10_names, fontsize=9)
axes[1].set_xlabel('AUC'); axes[1].set_xlim(0.5, 1.02)
axes[1].set_title('Top-10 Features by AUC', fontweight='bold')
axes[1].axvline(0.5, color='red', lw=0.8, linestyle='--')
axes[1].grid(axis='x', alpha=0.3)

# Legend
from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(color='#4CAF50', label='Alpha Asymmetry'),
    Patch(color='#2196F3', label='Spectral Power')
], fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(CFG['results_dir'], 'fig_feature_auc_ranking.png'),
             dpi=150)
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 10 — Monte Carlo 10-Fold CV × 100 Repeats
#
#  Protocol (matches paper exactly):
#    For each of 100 repeats:
#      Create fresh StratifiedKFold(10, random_state=repeat)
#      For each of 10 folds:
#        1. Fit StandardScaler on TRAIN fold only
#        2. Fit AUC ranking on TRAIN fold only → select top-N features
#        3. Train classifier on TRAIN fold
#        4. Evaluate on TEST fold (never seen during training)
#
#  Feature subsets evaluated: top-5 / top-10 / top-15 / top-19
#  Three classifiers: LR, SVM (linear), Naive Bayes
# ═══════════════════════════════════════════════════════════════════════════

def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return dict(
        acc  = accuracy_score(y_true, y_pred),
        sen  = tp / (tp + fn + 1e-9),
        spe  = tn / (tn + fp + 1e-9),
        f1   = f1_score(y_true, y_pred),
        auc  = roc_auc_score(y_true, y_prob),
    )


CLASSIFIERS = {
    'LR'  : LogisticRegression(max_iter=1000, random_state=SEED,
                                class_weight='balanced'),
    'SVM' : SVC(kernel='linear', probability=True,
                class_weight='balanced', random_state=SEED),
    'NB'  : GaussianNB(),
}


def run_monte_carlo_cv(F, y, cfg, classifiers=CLASSIFIERS):
    """
    Monte Carlo 10-fold CV × 100 repeats for each (classifier, top-N) pair.

    Returns
    -------
    results_df : DataFrame with columns:
                 [classifier, n_features, repeat, fold,
                  acc, sen, spe, f1, auc]
    """
    records = []
    n_subsets    = cfg['feature_subsets']
    n_repeats    = cfg['n_repeats']
    k_folds      = cfg['k_folds']
    clf_names    = list(classifiers.keys())
    total_iters  = n_repeats * k_folds * len(clf_names) * len(n_subsets)

    print(f'Total CV iterations: {total_iters:,}')
    print(f'  ({n_repeats} repeats × {k_folds} folds × '
          f'{len(clf_names)} classifiers × {len(n_subsets)} subsets)')

    pbar = tqdm(total=n_repeats, desc='  MC repeats')

    for repeat in range(n_repeats):
        skf = StratifiedKFold(n_splits=k_folds, shuffle=True,
                               random_state=repeat * 13 + SEED)

        for fold_i, (tr_idx, te_idx) in enumerate(skf.split(F, y)):
            F_tr, y_tr = F[tr_idx], y[tr_idx]
            F_te, y_te = F[te_idx], y[te_idx]

            # ── Step 1: Z-score normalize (fit on TRAIN only) ──────────
            scaler = StandardScaler()
            F_tr_n = scaler.fit_transform(F_tr)
            F_te_n = scaler.transform(F_te)

            # ── Step 2: AUC feature ranking (fit on TRAIN only) ────────
            ranked_idx, _ = auc_feature_ranking(F_tr_n, y_tr)

            # ── Step 3 & 4: Train + evaluate per (classifier, top-N) ──
            for n_top in n_subsets:
                top_idx    = ranked_idx[:n_top]
                F_tr_sel   = F_tr_n[:, top_idx]
                F_te_sel   = F_te_n[:, top_idx]

                for clf_name, clf_proto in classifiers.items():
                    clf = clone(clf_proto)
                    clf.fit(F_tr_sel, y_tr)
                    y_pred = clf.predict(F_te_sel)
                    y_prob = clf.predict_proba(F_te_sel)[:, 1]
                    m      = compute_metrics(y_te, y_pred, y_prob)
                    records.append({
                        'classifier' : clf_name,
                        'n_features' : n_top,
                        'repeat'     : repeat,
                        'fold'       : fold_i,
                        **m
                    })

        pbar.update(1)
    pbar.close()

    return pd.DataFrame(records)


print('Starting Monte Carlo CV...')
t0     = time.time()
df_raw = run_monte_carlo_cv(F_all, y_all, CFG)
print(f'\nCompleted in {(time.time()-t0)/60:.1f} min')
df_raw.to_csv(os.path.join(CFG['results_dir'], 'mc_cv_raw_results.csv'),
               index=False)
print('Raw results saved.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 11 — Results Summary Table (IEEE Format)
#  Mean ± Std across all 100 repeats × 10 folds = 1000 test evaluations
# ═══════════════════════════════════════════════════════════════════════════

metrics = ['acc', 'sen', 'spe', 'f1', 'auc']
metric_labs = ['Accuracy', 'Sensitivity', 'Specificity', 'F1-Score', 'AUC-ROC']

summary_rows = []
for (clf, n_feat), grp in df_raw.groupby(['classifier', 'n_features']):
    row = {'Classifier': clf, 'Top-N Features': n_feat}
    for m, ml in zip(metrics, metric_labs):
        row[ml] = f'{grp[m].mean()*100:.2f} ± {grp[m].std()*100:.2f}'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(['Classifier', 'Top-N Features'])

print('╔' + '═'*80 + '╗')
print('║  Monte Carlo 10-Fold CV × 100 Repeats  |  Mean ± Std (%)              ║')
print('║  Normalization + AUC ranking fitted on TRAIN fold only               ║')
print('╠' + '═'*80 + '╣')
print(summary_df.to_string(index=False))
print('╚' + '═'*80 + '╝')
print('\nPaper benchmark: SVM 98.4% | LR 97.6% | NB 96.8%  (alpha asymmetry only)')

summary_df.to_csv(os.path.join(CFG['results_dir'], 'summary_results.csv'),
                   index=False)

# ── Find best configuration per classifier ────────────────────────────────
print('\nBest top-N per classifier (by mean accuracy):')
for clf_name in ['LR', 'SVM', 'NB']:
    grp   = df_raw[df_raw['classifier'] == clf_name]
    best  = grp.groupby('n_features')['acc'].mean().idxmax()
    best_m = grp[grp['n_features'] == best]
    print(f'  {clf_name:<4}: top-{best:2d}  '
          f'Acc={best_m["acc"].mean()*100:.2f}%  '
          f'Sen={best_m["sen"].mean()*100:.2f}%  '
          f'Spe={best_m["spe"].mean()*100:.2f}%  '
          f'F1={best_m["f1"].mean()*100:.2f}%  '
          f'AUC={best_m["auc"].mean():.4f}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 12 — Figures (IEEE Paper Quality)
# ═══════════════════════════════════════════════════════════════════════════

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12})
COLORS = {'LR': '#2196F3', 'SVM': '#4CAF50', 'NB': '#FF9800'}

# ── Figure 1: Accuracy vs top-N features per classifier ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (clf_name, col) in zip(axes, COLORS.items()):
    grp   = df_raw[df_raw['classifier'] == clf_name]
    means = grp.groupby('n_features')['acc'].mean() * 100
    stds  = grp.groupby('n_features')['acc'].std()  * 100
    ax.errorbar(means.index, means.values, yerr=stds.values,
                 marker='o', lw=2, color=col, capsize=4)
    ax.set_title(clf_name, fontweight='bold')
    ax.set_xlabel('Number of Top Features')
    ax.set_xticks(CFG['feature_subsets'])
    ax.grid(alpha=0.3)
    # Mark best point
    best_n = means.idxmax()
    ax.axvline(best_n, color='red', linestyle='--', lw=1.2, alpha=0.7,
                label=f'Best: top-{best_n}')
    ax.legend(fontsize=9)
axes[0].set_ylabel('Accuracy (%)')
plt.suptitle('Accuracy vs Feature Subset Size  (MC 10-fold × 100)',
              fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CFG['results_dir'], 'fig_accuracy_vs_features.png'),
             dpi=150)
plt.show()

# ── Figure 2: Violin plot — accuracy distribution across 1000 evaluations ─
fig, ax = plt.subplots(figsize=(9, 5))
# Use best subset for each classifier
best_data = {}
for clf_name in ['LR', 'SVM', 'NB']:
    grp      = df_raw[df_raw['classifier'] == clf_name]
    best_n   = grp.groupby('n_features')['acc'].mean().idxmax()
    best_data[clf_name] = grp[grp['n_features'] == best_n]['acc'].values * 100

positions = [1, 2, 3]
vp = ax.violinplot([best_data[c] for c in ['LR','SVM','NB']],
                    positions=positions, showmedians=True, showextrema=True)
for body, (_, col) in zip(vp['bodies'], COLORS.items()):
    body.set_facecolor(col); body.set_alpha(0.7)
ax.set_xticks(positions)
ax.set_xticklabels(['LR', 'SVM (linear)', 'Naïve Bayes'])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy Distribution — Best Feature Subset per Classifier
'
              '(MC 10-fold × 100, 1000 test evaluations each)',
              fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG['results_dir'], 'fig_accuracy_violin.png'), dpi=150)
plt.show()

# ── Figure 3: Bar chart of all metrics at best configuration ──────────────
fig, ax = plt.subplots(figsize=(10, 4))
clfs       = ['LR', 'SVM', 'NB']
metric_sel = ['acc', 'sen', 'spe', 'f1']
ml_sel     = ['Accuracy', 'Sensitivity', 'Specificity', 'F1-Score']
x          = np.arange(len(clfs))
width      = 0.20
bar_colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for i, (m, ml) in enumerate(zip(metric_sel, ml_sel)):
    vals = []
    errs = []
    for clf_name in clfs:
        grp     = df_raw[df_raw['classifier'] == clf_name]
        best_n  = grp.groupby('n_features')['acc'].mean().idxmax()
        best_g  = grp[grp['n_features'] == best_n]
        vals.append(best_g[m].mean() * 100)
        errs.append(best_g[m].std()  * 100)
    ax.bar(x + (i - 1.5) * width, vals, width, yerr=errs, capsize=3,
            label=ml, color=bar_colors[i], edgecolor='black',
            alpha=0.85, linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(['LR', 'SVM (linear)', 'Naïve Bayes'])
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 115)
ax.set_title('Classifier Performance at Best Feature Subset  (Mean ± Std)',
              fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG['results_dir'], 'fig_classifier_comparison.png'),
             dpi=150)
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 13 — Feature Family Ablation
#  Compare: Spectral only vs Asymmetry only vs Combined
#  Tests the paper's claim that alpha asymmetry is the most reliable feature
# ═══════════════════════════════════════════════════════════════════════════

# Identify feature indices by family
spec_idx  = np.array([i for i, n in enumerate(feat_names) if n.startswith('BP_')])
asym_idx  = np.array([i for i, n in enumerate(feat_names) if n.startswith('Asym_')])

ABLATION = {
    'Spectral only'       : spec_idx,
    'Asymmetry only'      : asym_idx,
    'Combined (all)  ★'  : np.arange(len(feat_names)),
}

ablation_rows = []
n_ablation_repeats = 20   # reduced for speed; increase to 100 for publication

for subset_name, feat_idx in ABLATION.items():
    F_sub = F_all[:, feat_idx]
    for clf_name, clf_proto in CLASSIFIERS.items():
        accs = []
        for repeat in range(n_ablation_repeats):
            skf = StratifiedKFold(n_splits=CFG['k_folds'], shuffle=True,
                                   random_state=repeat * 13 + SEED)
            for tr_idx, te_idx in skf.split(F_sub, y_all):
                scaler  = StandardScaler()
                F_tr_n  = scaler.fit_transform(F_sub[tr_idx])
                F_te_n  = scaler.transform(F_sub[te_idx])
                clf     = clone(clf_proto)
                clf.fit(F_tr_n, y_all[tr_idx])
                accs.append(accuracy_score(
                    y_all[te_idx], clf.predict(F_te_n)
                ))
        ablation_rows.append({
            'Feature set'  : subset_name,
            'Classifier'   : clf_name,
            'Accuracy (%)'  : f'{np.mean(accs)*100:.2f} ± {np.std(accs)*100:.2f}',
            'Mean'          : np.mean(accs) * 100,
        })

df_abl = pd.DataFrame(ablation_rows)
pivot  = df_abl.pivot(index='Feature set', columns='Classifier',
                        values='Accuracy (%)')[['LR','SVM','NB']]

print('╔' + '═'*65 + '╗')
print('║  FEATURE FAMILY ABLATION  (MC 10-fold × 20 repeats)             ║')
print('╠' + '═'*65 + '╣')
print(pivot.to_string())
print('╚' + '═'*65 + '╝')
print('\n★ = Combined (full feature set)')

df_abl.to_csv(os.path.join(CFG['results_dir'], 'ablation_results.csv'),
               index=False)

# ── Grouped bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x      = np.arange(3)      # 3 classifiers
width  = 0.25
fsubs  = ['Spectral only', 'Asymmetry only', 'Combined (all)  ★']
colors = ['#90CAF9', '#A5D6A7', '#4CAF50']

for i, (fset, col) in enumerate(zip(fsubs, colors)):
    vals = [df_abl[(df_abl['Feature set']==fset) &
                    (df_abl['Classifier']==c)]['Mean'].values[0]
             for c in ['LR','SVM','NB']]
    ax.bar(x + (i-1)*width, vals, width, label=fset.replace('  ★',''),
            color=col, edgecolor='black', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(['LR', 'SVM (linear)', 'Naïve Bayes'])
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 115)
ax.set_title('Feature Family Ablation Study', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG['results_dir'], 'fig_ablation.png'), dpi=150)
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 14 — Statistical Significance Testing
#  Wilcoxon signed-rank test: pairwise classifier comparison
#  Tests whether accuracy differences between classifiers are significant
# ═══════════════════════════════════════════════════════════════════════════

from scipy.stats import wilcoxon

# Use best-subset accuracies across all 1000 evaluations per classifier
clf_accs = {}
for clf_name in ['LR', 'SVM', 'NB']:
    grp    = df_raw[df_raw['classifier'] == clf_name]
    best_n = grp.groupby('n_features')['acc'].mean().idxmax()
    clf_accs[clf_name] = grp[grp['n_features'] == best_n]['acc'].values

print('Wilcoxon signed-rank tests (pairwise, best feature subset):')
print('─' * 60)
pairs = [('SVM','LR'), ('SVM','NB'), ('LR','NB')]
for a, b in pairs:
    min_len = min(len(clf_accs[a]), len(clf_accs[b]))
    try:
        stat, pval = wilcoxon(clf_accs[a][:min_len], clf_accs[b][:min_len])
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else
               ('*' if pval < 0.05 else 'ns'))
    except Exception:
        pval, sig = np.nan, 'n/a'
    delta = (np.mean(clf_accs[a]) - np.mean(clf_accs[b])) * 100
    print(f'  {a} vs {b:<4}:  Δ={delta:+.3f}%   p={pval:.4f}   {sig}')

print('\n  *** p<0.001  ** p<0.01  * p<0.05  ns=not significant')
print('\n  NOTE: With 1000 samples the test is highly powered.')
print('         Report effect size (Δ%) alongside p-values in the paper.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 15 — Alpha Asymmetry Feature Deep-Dive
#  Visualise MDD vs HC distribution for each hemisphere pair
# ═══════════════════════════════════════════════════════════════════════════

asym_feat_idx = [i for i, n in enumerate(feat_names) if n.startswith('Asym_')]
asym_feat_names = [feat_names[i] for i in asym_feat_idx]
n_asym_feats    = len(asym_feat_idx)

fig, axes = plt.subplots(1, n_asym_feats, figsize=(4*n_asym_feats, 4))
if n_asym_feats == 1:
    axes = [axes]

for ax, fi, fname in zip(axes, asym_feat_idx, asym_feat_names):
    mdd_vals = F_all[y_all == 1, fi]
    hc_vals  = F_all[y_all == 0, fi]
    region   = fname.replace('Asym_alpha_', '').capitalize()

    ax.hist(mdd_vals, bins=15, alpha=0.65, color='tomato',
             label=f'MDD (n={len(mdd_vals)})', density=True)
    ax.hist(hc_vals,  bins=15, alpha=0.65, color='steelblue',
             label=f'HC  (n={len(hc_vals)})',  density=True)
    ax.axvline(0, color='black', lw=0.8, linestyle='--', alpha=0.5)
    ax.set_title(f'{region}\nAUC={global_auc_scores[fi]:.3f}',
                  fontsize=10, fontweight='bold')
    ax.set_xlabel('Asymmetry Index (R−L)/(R+L)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Density')
plt.suptitle('Alpha Interhemispheric Asymmetry: MDD vs HC  (raw, unnormalised)',
              fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CFG['results_dir'],
             'fig_alpha_asymmetry_distribution.png'), dpi=150)
plt.show()

# ── Significance test per pair ─────────────────────────────────────────────
from scipy.stats import mannwhitneyu
print('\nMann-Whitney U test per alpha asymmetry pair (MDD vs HC):')
print('─' * 52)
for fi, fname in zip(asym_feat_idx, asym_feat_names):
    mdd_v = F_all[y_all == 1, fi]
    hc_v  = F_all[y_all == 0, fi]
    stat, pval = mannwhitneyu(mdd_v, hc_v, alternative='two-sided')
    sig  = '***' if pval < 0.001 else ('**' if pval < 0.01 else
            ('*' if pval < 0.05 else 'ns'))
    region = fname.replace('Asym_alpha_','')
    print(f'  {region:<12}: MDD={mdd_v.mean():+.3f}  HC={hc_v.mean():+.3f}  '
          f'p={pval:.4f}  {sig}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CELL 16 — Final Summary & Reviewer Checklist
# ═══════════════════════════════════════════════════════════════════════════

# Best results per classifier
print('╔' + '═'*68 + '╗')
print('║  FINAL RESULTS SUMMARY                                             ║')
print('╠' + '═'*68 + '╣')
print(f'║  Subjects: MDD={np.sum(y_all==1)}, HC={np.sum(y_all==0)}, Total={len(y_all)}{" "*37}║')
print(f'║  Features: {len(feat_names)} total  '
      f'(Spectral={sum(1 for n in feat_names if n.startswith("BP_"))}, '
      f'Asymmetry={sum(1 for n in feat_names if n.startswith("Asym_"))})'
      f'{" "*15}║')
print(f'║  Validation: MC {CFG["n_repeats"]} repeats × {CFG["k_folds"]}-fold CV '
      f'= {CFG["n_repeats"]*CFG["k_folds"]} test evaluations per config{" "*12}║')
print('╠' + '═'*68 + '╣')
print('║  BEST RESULTS per classifier (paper benchmark in parentheses)      ║')

for clf_name, paper_acc in [('SVM', 98.4), ('LR', 97.6), ('NB', 96.8)]:
    grp    = df_raw[df_raw['classifier'] == clf_name]
    best_n = grp.groupby('n_features')['acc'].mean().idxmax()
    best_g = grp[grp['n_features'] == best_n]
    line   = (f'  {clf_name:<4} top-{best_n:2d}: '
              f'Acc={best_g["acc"].mean()*100:.2f}%  '
              f'Sen={best_g["sen"].mean()*100:.2f}%  '
              f'Spe={best_g["spe"].mean()*100:.2f}%  '
              f'(paper: {paper_acc}%)')
    print(f'║  {line:<66}║')

print('╠' + '═'*68 + '╣')
print('║  REVIEWER / Q1 CHECKLIST                                          ║')
items = [
    '□ Confirm preprocessing matches BESA-cleaned 120 s per subject',
    '□ AUC ranking verified inside fold — no leakage (Cell 10)',
    '□ Asymmetry ablation done (Cell 13): confirm asymmetry > spectral',
    '□ Statistical tests (Cell 14): Wilcoxon + Mann-Whitney per pair',
    '□ Report both EC and EO condition results if both are in data',
    '□ Increase ablation n_repeats to 100 before paper submission',
    '□ Plot confusion matrix for best SVM / LR / NB configuration',
    '□ Discuss small dataset (63 subjects) in limitations section',
]
for item in items:
    print(f'║  {item:<66}║')
print('╚' + '═'*68 + '╝')
